In [1]:
# ── Section 0: Imports & Config ──────────────────────────────────────────────
import json, os, re
from pathlib import Path

import jsonlines
import pandas as pd
from tqdm.notebook import tqdm

In [2]:

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_PATH   = "sample-1M.jsonl"
NER_OUT_PATH   = "ner_results.jsonl"        # one JSON line per article
PROGRESS_PATH  = "ner_progress.json"        # set of completed article IDs
EL_OUT_PATH    = "entity_linking_results.jsonl"

# ── GLiNER labels ────────────────────────────────────────────────────────────
LABELS = [
    "person", "norp", "facility", "organization", "gpe", "location",
    "product", "event", "work_of_art", "law", "language",
    "date", "time", "percent", "money", "quantity", "ordinal", "cardinal",
    "religion", "political_party", "nationality", "ethnic_group",
    "title", "award", "disease", "chemical", "weapon",
    "vehicle", "currency", "brand"
]

# ── Chunking ─────────────────────────────────────────────────────────────────
# GLiNER ~380 subword tokens ≈ ~280 words safely
# ── Section 0 config: replace word-based constants with token-based ones ──────
GLINER_MAX_TOKENS   = 380   # hard ceiling GLiNER accepts (384 minus a little margin)
GLINER_STRIDE_TOKENS = 60   # overlap in tokens so boundary entities aren't missed

GLINER_THRESHOLD  = 0.4   # entity confidence threshold

In [3]:
# ── Section 1: Load Models ────────────────────────────────────────────────────
from gliner import GLiNER
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# GLiNER
print("Loading GLiNER …")
gliner_model = GLiNER.from_pretrained("urchade/gliner_mediumv2.1")
gliner_model.to(DEVICE)

# GENRE  — facebook/genre-linking-aidayago2 is the standard EL checkpoint
# It links to Wikipedia titles given [START_ENT] … [END_ENT] context
print("Loading GENRE …")
GENRE_MODEL_NAME = "facebook/genre-linking-aidayago2"
genre_tokenizer  = AutoTokenizer.from_pretrained(GENRE_MODEL_NAME)
genre_model      = AutoModelForSeq2SeqLM.from_pretrained(GENRE_MODEL_NAME)
genre_model.to(DEVICE)
genre_model.eval()
print("Models ready.")

Using device: cpu
Loading GLiNER …


/Users/punreachrany/Desktop/University of Alberta/Knowledge Graph/long-tail-entities-news/kg/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading GENRE …


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Models ready.


In [4]:
# ── Section 2: GLiNER helpers ─────────────────────────────────────────────────

# ── Section 2: replace split_into_chunks with a token-aware version ───────────

def split_into_chunks(text: str) -> list[tuple[str, int]]:
    """
    Splits *text* into overlapping chunks that each stay within
    GLINER_MAX_TOKENS subword tokens — using GLiNER's own tokenizer
    so the count is exact and no truncation warnings fire.

    Returns list of (chunk_text, char_offset_in_original).
    """
    tokenizer = gliner_model.data_processor.transformer_tokenizer

    # Encode the whole document once to get the token↔char mapping
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False,   # GLiNER adds its own specials
        truncation=False,
    )

    tokens         = encoding["input_ids"]
    offset_mapping = encoding["offset_mapping"]   # (char_start, char_end) per token
    total_tokens   = len(tokens)

    chunks = []
    i = 0
    while i < total_tokens:
        end = min(i + GLINER_MAX_TOKENS, total_tokens)

        # Char span that covers tokens[i:end]
        char_start = offset_mapping[i][0]
        char_end   = offset_mapping[end - 1][1]
        chunk_text = text[char_start:char_end]

        chunks.append((chunk_text, char_start))

        if end == total_tokens:
            break
        i += GLINER_MAX_TOKENS - GLINER_STRIDE_TOKENS   # slide forward with stride

    return chunks


def resolve_overlaps(entities: list[dict]) -> list[dict]:
    """
    When entity A's span is fully contained within entity B's span,
    keep B (the longer / more specific mention) and discard A.
    Example: 'Alberta' inside 'University of Alberta' → keep the latter.
    """
    # Sort by span length descending so longer spans are checked first
    sorted_ents = sorted(entities, key=lambda e: e["end"] - e["start"], reverse=True)
    kept = []
    for cand in sorted_ents:
        dominated = any(
            (k["start"] <= cand["start"] and k["end"] >= cand["end"]
             and not (k["start"] == cand["start"] and k["end"] == cand["end"]))
            for k in kept
        )
        if not dominated:
            kept.append(cand)
    return kept


def run_gliner_on_text(text: str) -> list[dict]:
    """
    Runs GLiNER on *text* with chunking, remaps offsets to the full document,
    resolves overlapping spans, and returns a deduplicated entity list.
    Each entity dict: {text, label, start, end, score}
    """
    chunks   = split_into_chunks(text)
    all_ents = []

    for chunk_text, char_offset in chunks:
        preds = gliner_model.predict_entities(
            chunk_text, LABELS, threshold=GLINER_THRESHOLD
        )
        for ent in preds:
            all_ents.append({
                "text":  ent["text"],
                "label": ent["label"],
                "start": ent["start"] + char_offset,   # remap to doc offset
                "end":   ent["end"]   + char_offset,
                "score": round(ent["score"], 4),
            })

    # Remove span-level duplicates from chunk overlaps
    seen = set()
    unique_ents = []
    for e in all_ents:
        key = (e["start"], e["end"], e["label"])
        if key not in seen:
            seen.add(key)
            unique_ents.append(e)

    return resolve_overlaps(unique_ents)

In [5]:
# ── Section 3: NER with progress tracking ────────────────────────────────────

def load_progress() -> set:
    """Load the set of already-processed article IDs."""
    if Path(PROGRESS_PATH).exists():
        with open(PROGRESS_PATH) as f:
            return set(json.load(f))
    return set()


def save_progress(done_ids: set):
    with open(PROGRESS_PATH, "w") as f:
        json.dump(list(done_ids), f)


def run_ner_pipeline(max_articles: int | None = None):
    """
    Streams through the dataset, skips already-processed IDs,
    writes NER results line-by-line to NER_OUT_PATH.
    Safe to interrupt and resume at any time.
    """
    done_ids = load_progress()
    print(f"Resuming: {len(done_ids)} articles already processed.")

    # Count total for the progress bar (optional — comment out if dataset is huge)
    with jsonlines.open(DATASET_PATH) as reader:
        total = sum(1 for _ in reader)

    processed = 0
    with jsonlines.open(DATASET_PATH) as reader, \
         jsonlines.open(NER_OUT_PATH, mode="a") as writer:

        for article in tqdm(reader, total=total, desc="NER"):
            art_id = article["id"]

            if art_id in done_ids:
                continue   # already done — skip

            # Combine title + content for richer context
            full_text = (article.get("title") or "") + " " + \
                        (article.get("content") or "")
            full_text  = full_text.strip()

            if not full_text:
                done_ids.add(art_id)
                continue

            entities = run_gliner_on_text(full_text)

            writer.write({
                "id":       art_id,
                "entities": entities,
            })

            done_ids.add(art_id)
            processed += 1

            # Flush progress every 100 articles so interrupts lose little work
            if processed % 100 == 0:
                save_progress(done_ids)

            if max_articles and processed >= max_articles:
                print(f"Stopped early after {processed} articles.")
                break

    save_progress(done_ids)
    print(f"Done. Total processed this run: {processed}")


# ── Run it (remove max_articles to process everything) ───────────────────────
run_ner_pipeline(max_articles=None)   # set None for the full dataset

Resuming: 8100 articles already processed.


NER:   0%|          | 0/1000000 [00:00<?, ?it/s]

/Users/punreachrany/Desktop/University of Alberta/Knowledge Graph/long-tail-entities-news/kg/lib/python3.13/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 385 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Users/punreachrany/Desktop/University of Alberta/Knowledge Graph/long-tail-entities-news/kg/lib/python3.13/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 392 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Users/punreachrany/Desktop/University of Alberta/Knowledge Graph/long-tail-entities-news/kg/lib/python3.13/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 397 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Users/punreachrany/Desktop/

KeyboardInterrupt: 

In [6]:
# ── Section 4: Deduplicate NER results ───────────────────────────────────────
# Dedup key = (normalized_surface_form, entity_type)
# We keep the highest-confidence occurrence of each unique (text, label) pair.

def load_ner_results(path: str = NER_OUT_PATH) -> list[dict]:
    results = []
    with jsonlines.open(path) as reader:
        for row in reader:
            results.append(row)
    return results


def build_dedup_entity_table(ner_results: list[dict]) -> pd.DataFrame:
    """
    Flatten all entities across all articles into one DataFrame,
    then deduplicate on (normalized_text, label).
    Returns the deduplicated DataFrame ready for entity linking.
    """
    rows = []
    for record in ner_results:
        art_id = record["id"]
        for ent in record["entities"]:
            rows.append({
                "article_id":  art_id,
                "text":        ent["text"],
                "norm_text":   ent["text"].strip().lower(),
                "label":       ent["label"],
                "start":       ent["start"],
                "end":         ent["end"],
                "score":       ent["score"],
            })

    df = pd.DataFrame(rows)

    # Keep the highest-score row per (norm_text, label) pair
    df_dedup = (
        df.sort_values("score", ascending=False)
          .drop_duplicates(subset=["norm_text", "label"])
          .reset_index(drop=True)
    )

    print(f"Total entity mentions : {len(df):,}")
    print(f"Unique (text, type)   : {len(df_dedup):,}")
    return df_dedup


ner_results = load_ner_results()
entity_df   = build_dedup_entity_table(ner_results)
entity_df.head(10)

Total entity mentions : 8,499,579
Unique (text, type)   : 1,722,591


,article_id,text,norm_text,label,start,end,score
0,043c3bbc-746a-4f3a-9967-d5bc62ceda34,gout,gout,disease,424,428,0.9981
1,43559af7-d09a-40a4-a7b2-3b82654e829c,Calvin Klein,calvin klein,brand,293,305,0.9980
2,23ab00fe-74fa-4c03-9558-52692e9d07e2,Burning Man,burning man,event,162,173,0.9978
3,89ad3773-189a-4c6f-bdc4-916533d8ca15,zara,zara,brand,241,245,0.9978
4,e092bc17-fb87-4cb1-af61-7672b2e9697c,Lyme Disease,lyme disease,disease,221,233,0.9978
5,1d07f3c0-51f7-4bcc-9b8a-772db958f524,Gaucher disease,gaucher disease,disease,3123,3138,0.9975
6,698434fa-c2d8-486a-b6dc-9754504999fd,Wounded Warrior Project,wounded warrior project,organization,127,150,0.9975
7,63192b2a-08cf-4645-969f-c87604a9a226,Aston Martin,aston martin,brand,25,37,0.9975
8,3a9f75fb-b854-4dac-a7e2-e7b752a63513,Gaelic,gaelic,language,1768,1774,0.9974
9,35bdaf75-aa95-4807-a4b0-6e0db5e28e0e,Zoya,zoya,brand,373,377,0.9974


In [ ]:
# ── Section 5: Entity Linking with GENRE ─────────────────────────────────────
# GENRE takes:  "[START_ENT] surface_form [END_ENT] surrounding_context"
# and generates the Wikipedia page title as the entity identifier.

GENRE_BATCH_SIZE = 16
GENRE_NUM_BEAMS  = 5


def build_genre_input(surface: str, context: str = "") -> str:
    """
    Wraps the surface form with GENRE's special markers.
    Optionally include a short context window for disambiguation.
    """
    if context:
        # Try to locate surface in context to surround it
        idx = context.lower().find(surface.lower())
        if idx != -1:
            prefix = context[max(0, idx - 100): idx]
            suffix = context[idx + len(surface): idx + len(surface) + 100]
            return f"{prefix} [START_ENT] {surface} [END_ENT] {suffix}"
    return f"[START_ENT] {surface} [END_ENT]"


def genre_link_batch(surfaces: list[str],
                     contexts: list[str] | None = None) -> list[str]:
    """
    Links a batch of surface forms to Wikipedia titles.
    Returns a list of predicted titles (empty string if failed).
    """
    if contexts is None:
        contexts = [""] * len(surfaces)

    inputs_text = [build_genre_input(s, c) for s, c in zip(surfaces, contexts)]
    inputs = genre_tokenizer(
        inputs_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = genre_model.generate(
            **inputs,
            num_beams=GENRE_NUM_BEAMS,
            num_return_sequences=1,
            max_length=64,
        )

    decoded = genre_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return decoded


def run_entity_linking(df: pd.DataFrame, limit: int | None = None) -> pd.DataFrame:
    """
    Runs GENRE over all unique entities in df.
    Adds a 'wiki_title' column with the predicted Wikipedia title.
    Set limit=N to only process the first N rows (useful for testing).
    """
    df = df.copy()
    if limit is not None:
        df = df.head(limit)

    surfaces    = df["text"].tolist()
    wiki_titles = []

    for i in tqdm(range(0, len(surfaces), GENRE_BATCH_SIZE), desc="GENRE linking"):
        batch = surfaces[i: i + GENRE_BATCH_SIZE]
        preds = genre_link_batch(batch)
        wiki_titles.extend(preds)

    df["wiki_title"] = wiki_titles
    return df


# Test on first 10
entity_df_linked = run_entity_linking(entity_df)

# Full run — just remove the limit
# entity_df_linked = run_entity_linking(entity_df)

entity_df_linked.head(10)

GENRE linking:   0%|          | 0/107662 [00:00<?, ?it/s]

In [ ]:
# ── Section 6: Save results ───────────────────────────────────────────────────

# Save the full linked entity table as CSV (human-readable, easy to open)
entity_df_linked.to_csv("entity_linking_results.csv", index=False)

# Also save as JSONL for downstream processing
with jsonlines.open(EL_OUT_PATH, mode="w") as writer:
    for _, row in entity_df_linked.iterrows():
        writer.write(row.to_dict())

print("Saved:")
print(f"  CSV   → entity_linking_results.csv  ({len(entity_df_linked):,} rows)")
print(f"  JSONL → {EL_OUT_PATH}")

# Quick summary: top entity types
print("\nTop entity types:")
print(entity_df_linked["label"].value_counts().head(15).to_string())